# SF-MT-MARL: Safety-Filtered Multi-Timescale Multi-Agent Reinforcement Learning

**Integrated CO₂ capture, pipeline transport, and CO₂-EOR operations**

This notebook provides a runnable research implementation of the framework described in the manuscript. The manuscript specifies three agents (capture, pipeline/compression, reservoir), multi-timescale decisions, inter-stage line-pack coordination, a safety action-projection layer, centralized training/decentralized execution, and a joint carbon/economic/safety reward. fileciteturn0file0L506-L511

**Important:** the manuscript does not provide an actual experimental time-series dataset or complete numerical plant parameters. It explicitly states that the experimental dataset *will be generated* from process simulation and public CO₂-EOR information. Therefore, this notebook uses a transparent physics-inspired surrogate environment with synthetic disturbances for reproducible algorithm development; its results must not be presented as field/plant validation. fileciteturn0file0L515-L539

In [ ]:
# ============================================================
# 1. INSTALL / IMPORTS / REPRODUCIBILITY
# ============================================================
!pip -q install numpy pandas matplotlib scikit-learn torch openpyxl

import os, math, random, warnings, json, gc, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import deque
from dataclasses import dataclass
from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

OUT_DIR = "/content/SFMT_MARL_Results"
os.makedirs(OUT_DIR, exist_ok=True)


## 2. System definition

The implementation follows the manuscript's three-agent decomposition: capture controls solvent circulation, regeneration heat and capture rate; pipeline controls compressor operation, inlet pressure and transported flow; reservoir controls injection rate and well allocation. fileciteturn0file0L600-L614

Decision multipliers are used to represent faster capture/compression, intermediate pipeline, and slower reservoir decisions. fileciteturn0file0L615-L629

In [ ]:
# ============================================================
# 2. CONFIGURATION
# ============================================================
@dataclass
class Config:
    dt: float = 1.0
    episode_steps: int = 240
    capture_interval: int = 1
    pipeline_interval: int = 2
    reservoir_interval: int = 6
    gamma: float = 0.98
    tau: float = 0.01
    actor_lr: float = 2e-4
    critic_lr: float = 5e-4
    batch_size: int = 128
    replay_size: int = 50000
    warmup: int = 1000
    train_steps: int = 1
    hidden: int = 128

CFG = Config()

# Physical-inspired limits
LIMITS = {
    "capture_rate": (60.0, 120.0),       # tCO2/h
    "solvent_flow": (0.60, 1.40),        # normalized plant units
    "regen_heat": (0.70, 1.40),
    "compressor_speed": (0.55, 1.15),
    "inlet_pressure": (75.0, 115.0),     # bar
    "pipeline_pressure": (70.0, 120.0),  # bar
    "co2_purity": (0.95, 0.999),
    "injection_rate": (50.0, 105.0),     # tCO2/h
    "bhp": (160.0, 285.0),               # bar
    "fracture_pressure": 300.0,
    "linepack": (500.0, 1800.0),          # tCO2
    "pipeline_capacity": 125.0,
}

ACTION_LOW = np.array([-1.0]*3, dtype=np.float32)
ACTION_HIGH = np.array([1.0]*3, dtype=np.float32)

# Local observations
OBS_DIMS = [10, 10, 10]
ACT_DIMS = [3, 3, 2]
GLOBAL_DIM = sum(OBS_DIMS) + 6


In [ ]:
# ============================================================
# 3. SYNTHETIC PHYSICS-INSPIRED INTEGRATED ENVIRONMENT
# ============================================================
class IntegratedCCUSEnv:
    def __init__(self, cfg=CFG, seed=SEED):
        self.cfg = cfg
        self.rng = np.random.default_rng(seed)
        self.t = 0
        self.reset()

    def reset(self):
        self.t = 0
        self.feed = 100.0
        self.feed_co2 = 0.18
        self.elec_price = 70.0
        self.oil_price = 75.0
        self.impurity = 0.02
        self.injectivity = 1.0

        self.capture_rate = 88.0
        self.solvent = 1.0
        self.regen = 1.0
        self.comp_speed = 0.85
        self.inlet_p = 92.0
        self.pipe_p = 95.0
        self.purity = 0.985
        self.linepack = 1000.0
        self.injection = 78.0
        self.bhp = 215.0
        self.oil_prod = 100.0
        self.prev_actions = [np.zeros(3), np.zeros(3), np.zeros(2)]
        return self._observations()

    def _disturbance(self):
        # Smooth operational variation + occasional events
        self.feed = 100 + 12*np.sin(self.t/30) + self.rng.normal(0, 2.0)
        self.feed_co2 = np.clip(0.18 + 0.025*np.sin(self.t/40) + self.rng.normal(0, .006), .10, .25)
        self.elec_price = np.clip(70 + 18*np.sin(self.t/50) + self.rng.normal(0, 4), 20, 140)
        self.oil_price = np.clip(75 + 12*np.sin(self.t/70) + self.rng.normal(0, 3), 35, 120)
        self.impurity = np.clip(.02 + .012*np.sin(self.t/27) + self.rng.normal(0, .003), .005, .08)

        # Scheduled stress events
        if 80 <= self.t < 105:       # compressor derating
            comp_factor = .78
        else:
            comp_factor = 1.0

        if 150 <= self.t < 180:      # reduced injectivity
            inj_factor = .68
        else:
            inj_factor = 1.0
        self.injectivity = inj_factor
        return comp_factor

    def _norm(self, x, lo, hi):
        return float(np.clip(2*(x-lo)/(hi-lo)-1, -1, 1))

    def _observations(self):
        d = np.array([
            self._norm(self.feed, 70, 130),
            self._norm(self.feed_co2, .08, .28),
            self._norm(self.elec_price, 10, 150),
            self._norm(self.oil_price, 20, 130),
            self._norm(self.impurity, 0, .10),
            self._norm(self.injectivity, .6, 1.05),
        ], dtype=np.float32)

        cap = np.array([
            self._norm(self.feed,70,130), self._norm(self.feed_co2,.08,.28),
            self._norm(self.capture_rate,40,130), self._norm(self.solvent,.4,1.6),
            self._norm(self.regen,.4,1.6), self._norm(self.elec_price,10,150),
            self._norm(self.impurity,0,.10), self._norm(self.linepack,300,2000),
            self._norm(self.injection,30,120), self._norm(self.purity,.90,1.0)
        ], dtype=np.float32)

        pipe = np.array([
            self._norm(self.capture_rate,40,130), self._norm(self.comp_speed,.4,1.3),
            self._norm(self.inlet_p,60,125), self._norm(self.pipe_p,60,130),
            self._norm(self.linepack,300,2000), self._norm(self.purity,.90,1.0),
            self._norm(self.injection,30,120), self._norm(self.elec_price,10,150),
            self._norm(self.impurity,0,.10), self._norm(self.injectivity,.6,1.05)
        ], dtype=np.float32)

        res = np.array([
            self._norm(self.injection,30,120), self._norm(self.bhp,130,310),
            self._norm(self.oil_prod,50,150), self._norm(self.linepack,300,2000),
            self._norm(self.capture_rate,40,130), self._norm(self.pipe_p,60,130),
            self._norm(self.oil_price,20,130), self._norm(self.injectivity,.6,1.05),
            self._norm(self.purity,.90,1.0), self._norm(self.elec_price,10,150)
        ], dtype=np.float32)
        return [cap, pipe, res]

    def _global(self, obs):
        return np.concatenate(obs + [
            np.array([
                self._norm(self.linepack,300,2000),
                self._norm(self.pipe_p,60,130),
                self._norm(self.bhp,130,310),
                self._norm(self.capture_rate,40,130),
                self._norm(self.injection,30,120),
                self._norm(self.elec_price,10,150)
            ], dtype=np.float32)
        ])

    def safety_filter(self, actions):
        # actions are [-1,1]; projection converts them to physical commands
        a0, a1, a2 = [np.asarray(a).copy() for a in actions]

        cap_rate = 90 + 35*a0[2]
        solvent = 1.0 + .45*a0[0]
        regen = 1.0 + .45*a0[1]

        comp = .85 + .30*a1[0]
        inlet = 95 + 28*a1[1]
        flow = 88 + 38*a1[2]

        inj = 78 + 42*a2[0]
        alloc = .5 + .5*a2[1]

        # Feasible action projection: downstream capacity limits upstream flow
        max_flow = min(LIMITS["pipeline_capacity"], 110*self.injectivity,
                       cap_rate, 50 + 65*comp)

        safe_flow = np.clip(flow, 55, max_flow)
        safe_cap = np.clip(cap_rate, 60, 120)
        safe_inj = np.clip(inj, 50, 105*self.injectivity)

        # Pressure/purity surrogate checks
        pred_pipe = self.pipe_p + .08*(safe_flow-self.injection) + 8*(comp-.85) + self.rng.normal(0,.15)
        pred_pipe = np.clip(pred_pipe, LIMITS["pipeline_pressure"][0], LIMITS["pipeline_pressure"][1])

        pred_purity = np.clip(.995 - 0.35*self.impurity - .003*abs(safe_flow-85), .95, .999)
        pred_bhp = self.bhp + .55*(safe_inj-self.injection) + 4*(alloc-.5)
        pred_bhp = min(pred_bhp, LIMITS["fracture_pressure"]*0.95)

        # Re-encode safe values as actions
        safe_a0 = np.array([
            (solvent-1)/.45, (regen-1)/.45, (safe_cap-90)/35
        ], dtype=np.float32)
        safe_a1 = np.array([
            (comp-.85)/.30, (inlet-95)/28, (safe_flow-88)/38
        ], dtype=np.float32)
        safe_a2 = np.array([
            (safe_inj-78)/42, (alloc-.5)/.5
        ], dtype=np.float32)

        correction = np.mean(np.abs(np.concatenate([a0-safe_a0, a1-safe_a1, a2-safe_a2])))
        return [np.clip(safe_a0,-1,1), np.clip(safe_a1,-1,1), np.clip(safe_a2,-1,1)], {
            "pipe_pressure_pred": pred_pipe,
            "purity_pred": pred_purity,
            "bhp_pred": pred_bhp,
            "correction": correction
        }

    def step(self, actions):
        comp_factor = self._disturbance()

        safe_actions, safety = self.safety_filter(actions)
        a0,a1,a2 = safe_actions

        self.solvent = np.clip(1+.45*a0[0], .6, 1.4)
        self.regen = np.clip(1+.45*a0[1], .7, 1.4)
        self.capture_rate = np.clip(90+35*a0[2], 60, 120) * (0.75+0.25*self.feed/100) * (1-0.25*self.impurity)

        self.comp_speed = np.clip((.85+.30*a1[0])*comp_factor, .55, 1.15)
        self.inlet_p = np.clip(95+28*a1[1], 75, 115)

        transported = np.clip(88+38*a1[2], 55, 125)
        self.injection = np.clip(78+42*a2[0], 50, 105*self.injectivity)

        # Line-pack mass balance from manuscript
        vent = max(0.0, transported-self.injection-self.linepack*.003)
        self.linepack += self.capture_rate-transported-vent
        self.linepack = np.clip(self.linepack, 300, 2000)

        self.pipe_p = np.clip(
            92 + .035*(self.linepack-1000) + .20*(transported-85) + 12*(self.comp_speed-.85),
            65, 125
        )
        self.purity = np.clip(.995 - .35*self.impurity - .0015*abs(transported-self.capture_rate), .93, .999)
        self.bhp = np.clip(
            215 + .65*(self.injection-78) + 10*(self.injectivity-1),
            140, 310
        )

        self.oil_prod = np.clip(
            100 + .22*(self.injection-70) + .08*(self.bhp-210) + self.rng.normal(0,1.0),
            50, 155
        )

        # Constraint magnitudes
        violations = {
            "mass_balance": abs(self.capture_rate-transported-self.linepack*.003-self.injection-vent)/max(self.capture_rate,1),
            "pipeline_pressure": max(0, self.pipe_p-120)/20 + max(0,70-self.pipe_p)/20,
            "purity": max(0,.95-self.purity)/.05,
            "fracture": max(0,self.bhp-285)/15,
            "flow": max(0,transported-125)/20,
        }
        violation = float(sum(violations.values()))

        # Accounting
        retained = max(0, self.injection*(.82 + .08*self.injectivity))
        energy = 0.018*self.capture_rate*self.regen**2 + 0.045*transported*(self.comp_speed**2)
        cost = energy*self.elec_price/100 + 0.015*self.capture_rate
        energy_emissions = 0.00038*energy*self.elec_price
        net_carbon = retained - vent - energy_emissions

        # Normalized shared reward
        reward = (
            1.20*(net_carbon/100)
            + 0.20*((self.oil_prod-90)/20)
            - 0.35*(energy/5)
            - 0.15*(cost/2)
            - 3.00*violation
            - 0.30*safety["correction"]
        )

        self.prev_actions = [a0.copy(), a1.copy(), a2.copy()]
        self.t += 1
        done = self.t >= self.cfg.episode_steps

        obs = self._observations()
        info = {
            "net_carbon": net_carbon, "retained": retained, "vent": vent,
            "energy": energy, "cost": cost, "oil": self.oil_prod,
            "violation": violation, "correction": safety["correction"],
            "pipe_pressure": self.pipe_p, "purity": self.purity,
            "bhp": self.bhp, "linepack": self.linepack
        }
        return obs, float(reward), done, info

    def global_state(self, obs):
        return self._global(obs)

env = IntegratedCCUSEnv()
obs = env.reset()
print([x.shape for x in obs], "global:", env.global_state(obs).shape)


## 4. Actor–critic networks

Centralized critics receive the global state and joint actions, while each actor receives its local observation. This implements the manuscript's centralized-training/decentralized-execution principle. fileciteturn0file0L680-L694

In [ ]:
# ============================================================
# 4. ACTOR / CENTRALIZED CRITIC
# ============================================================
class Actor(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, act_dim), nn.Tanh()
        )
    def forward(self, x):
        return self.net(x)

class CentralCritic(nn.Module):
    def __init__(self, global_dim, joint_act_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(global_dim + joint_act_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1)
        )
    def forward(self, state, actions):
        return self.net(torch.cat([state, actions], dim=-1))

actors = [Actor(OBS_DIMS[i], ACT_DIMS[i], CFG.hidden).to(DEVICE) for i in range(3)]
critics = [CentralCritic(GLOBAL_DIM, sum(ACT_DIMS), CFG.hidden).to(DEVICE) for _ in range(3)]
target_actors = [Actor(OBS_DIMS[i], ACT_DIMS[i], CFG.hidden).to(DEVICE) for i in range(3)]
target_critics = [CentralCritic(GLOBAL_DIM, sum(ACT_DIMS), CFG.hidden).to(DEVICE) for _ in range(3)]

for i in range(3):
    target_actors[i].load_state_dict(actors[i].state_dict())
    target_critics[i].load_state_dict(critics[i].state_dict())

actor_opts = [optim.Adam(a.parameters(), lr=CFG.actor_lr) for a in actors]
critic_opts = [optim.Adam(c.parameters(), lr=CFG.critic_lr) for c in critics]


In [ ]:
# ============================================================
# 5. REPLAY BUFFER + MULTI-TIMESCALE CONTROLLER
# ============================================================
class ReplayBuffer:
    def __init__(self, capacity):
        self.data = deque(maxlen=capacity)
    def add(self, s, a, r, ns, done):
        self.data.append((s,a,r,ns,done))
    def __len__(self):
        return len(self.data)
    def sample(self, n):
        batch = random.sample(self.data, n)
        s,a,r,ns,d = zip(*batch)
        return (np.asarray(s,np.float32), np.asarray(a,np.float32),
                np.asarray(r,np.float32), np.asarray(ns,np.float32),
                np.asarray(d,np.float32))

buffer = ReplayBuffer(CFG.replay_size)

class MultiTimescaleController:
    def __init__(self):
        self.last = [np.zeros(3,np.float32), np.zeros(3,np.float32), np.zeros(2,np.float32)]
    def select(self, actors, obs, step, noise=.15):
        intervals = [CFG.capture_interval, CFG.pipeline_interval, CFG.reservoir_interval]
        for i in range(3):
            if step % intervals[i] == 0:
                x = torch.tensor(obs[i],dtype=torch.float32,device=DEVICE).unsqueeze(0)
                with torch.no_grad():
                    a = actors[i](x).cpu().numpy()[0]
                a += noise*np.random.randn(len(a))
                self.last[i] = np.clip(a,-1,1).astype(np.float32)
        return [x.copy() for x in self.last]

controller = MultiTimescaleController()


In [ ]:
# ============================================================
# 6. TRAINING UTILITIES
# ============================================================
def soft_update(source, target, tau):
    for p, tp in zip(source.parameters(), target.parameters()):
        tp.data.copy_(tau*p.data + (1-tau)*tp.data)

def train_agents():
    if len(buffer) < max(CFG.batch_size, CFG.warmup):
        return None

    s,a,r,ns,d = buffer.sample(CFG.batch_size)
    s = torch.tensor(s,device=DEVICE)
    a = torch.tensor(a,device=DEVICE)
    r = torch.tensor(r,device=DEVICE).unsqueeze(1)
    ns = torch.tensor(ns,device=DEVICE)
    d = torch.tensor(d,device=DEVICE).unsqueeze(1)

    # Split global state into local observations
    slices=[]
    st=0
    for dim in OBS_DIMS:
        slices.append((st,st+dim)); st+=dim
    next_local = [ns[:,lo:hi] for lo,hi in slices]

    losses={}
    for i in range(3):
        # target joint action
        next_actions=[]
        for j in range(3):
            x=next_local[j]
            next_actions.append(target_actors[j](x))
        next_joint=torch.cat(next_actions,dim=1)
        with torch.no_grad():
            y=r + CFG.gamma*(1-d)*target_critics[i](ns,next_joint)

        q=critics[i](s,a)
        critic_loss=F.mse_loss(q,y)

        critic_opts[i].zero_grad()
        critic_loss.backward()
        nn.utils.clip_grad_norm_(critics[i].parameters(), 5)
        critic_opts[i].step()

        # Current actor i, other actions treated as constants
        local=s[:,slices[i][0]:slices[i][1]]
        ai=actors[i](local)
        joint=[]
        for j in range(3):
            if j==i: joint.append(ai)
            else:
                lo,hi=slices[j]
                joint.append(a[:,sum(ACT_DIMS[:j]):sum(ACT_DIMS[:j+1])].detach())
        joint=torch.cat(joint,dim=1)
        actor_loss=-critics[i](s,joint).mean()

        actor_opts[i].zero_grad()
        actor_loss.backward()
        nn.utils.clip_grad_norm_(actors[i].parameters(), 5)
        actor_opts[i].step()

        soft_update(actors[i],target_actors[i],CFG.tau)
        soft_update(critics[i],target_critics[i],CFG.tau)
        losses[f"critic_{i}"]=critic_loss.item()
        losses[f"actor_{i}"]=actor_loss.item()
    return losses


## 7. Training

The environment includes nominal operation plus feed variation, electricity/oil price changes, compressor derating, reduced injectivity and stochastic sensor/process noise. These correspond to the disturbance types described in the manuscript. fileciteturn0file0L89-L93

In [ ]:
# ============================================================
# 7. TRAIN SF-MT-MARL
# ============================================================
NUM_EPISODES = 35
train_history=[]

for ep in range(NUM_EPISODES):
    obs=env.reset()
    controller=MultiTimescaleController()
    ep_return=0.0
    ep_violation=0.0
    ep_corr=0.0

    for t in range(CFG.episode_steps):
        state=env.global_state(obs)
        actions=controller.select(actors,obs,t,noise=max(.04,.20*(1-ep/NUM_EPISODES)))
        safe_obs,r,done,info=env.step(actions)
        next_state=env.global_state(safe_obs)

        joint_action=np.concatenate(actions)
        buffer.add(state,joint_action,r,next_state,float(done))
        obs=safe_obs

        ep_return+=r
        ep_violation+=info["violation"]
        ep_corr+=info["correction"]

        if len(buffer)>=CFG.warmup:
            for _ in range(CFG.train_steps):
                train_agents()
        if done: break

    train_history.append([ep+1,ep_return,ep_violation/CFG.episode_steps,
                          ep_corr/CFG.episode_steps])
    print(f"Episode {ep+1:03d}/{NUM_EPISODES} | Return {ep_return:8.2f} | "
          f"Mean violation {ep_violation/CFG.episode_steps:.4f}")

train_df=pd.DataFrame(train_history,columns=["Episode","Return","MeanViolation","MeanSafetyCorrection"])
train_df.to_csv(os.path.join(OUT_DIR,"training_history.csv"),index=False)


## 8. Evaluation and reference controllers

The reference controller is a deterministic rule-based coordinated controller. The comparison is descriptive: it measures the simulated environment under the same scenario seeds and constraints. It is not a claim of superiority in a real facility.

In [ ]:
# ============================================================
# 8. EVALUATION FUNCTIONS
# ============================================================
def proposed_action(obs, step, noise=0.0):
    return controller.select(actors,obs,step,noise=noise)

def rule_based_action(env):
    # Conservative local coordination controller
    cap = np.clip((env.feed*env.feed_co2 - 85)/35, -1, 1)
    pipe = np.clip((100-env.linepack)/500, -1, 1)
    inj = np.clip((85-env.injection)/35, -1, 1)
    return [
        np.array([0.0, 0.0, cap],np.float32),
        np.array([0.0, 0.0, pipe],np.float32),
        np.array([inj, 0.0],np.float32)
    ]

def evaluate(policy, episodes=8, seed_offset=1000):
    rows=[]
    for e in range(episodes):
        ev=IntegratedCCUSEnv(seed=seed_offset+e)
        obs=ev.reset()
        ctrl=MultiTimescaleController()
        totals={k:0.0 for k in ["reward","net_carbon","retained","vent","energy","cost","oil","violation","correction"]}
        for t in range(CFG.episode_steps):
            if policy=="proposed":
                actions=ctrl.select(actors,obs,t,noise=0)
            else:
                actions=rule_based_action(ev)
            obs,r,done,info=ev.step(actions)
            totals["reward"]+=r
            for k in totals:
                if k not in ["reward"]: totals[k]+=info[k]
        rows.append({"Episode":e+1,**totals})
    return pd.DataFrame(rows)

proposed_df=evaluate("proposed")
baseline_df=evaluate("baseline")

def summarize(df,name):
    return pd.Series({
        "Controller":name,
        "Return":df.reward.mean(),
        "Net CO2 retained":df.net_carbon.mean(),
        "CO2 retained":df.retained.mean(),
        "Vented CO2":df.vent.mean(),
        "Energy":df.energy.mean(),
        "Operating cost":df.cost.mean(),
        "Oil production":df.oil.mean(),
        "Safety violation":df.violation.mean(),
        "Safety correction":df.correction.mean()
    })

comparison=pd.DataFrame([summarize(proposed_df,"SF-MT-MARL"),
                         summarize(baseline_df,"Reference controller")])
comparison.to_csv(os.path.join(OUT_DIR,"controller_comparison.csv"),index=False)
display(comparison)


In [ ]:
# ============================================================
# 9. MULTI-SCENARIO STRESS TESTS
# ============================================================
def scenario_run(name, modify):
    ev=IntegratedCCUSEnv(seed=500)
    obs=ev.reset()
    ctrl=MultiTimescaleController()
    totals={k:0.0 for k in ["reward","net_carbon","retained","vent","energy","cost","oil","violation","correction"]}
    trace=[]
    for t in range(CFG.episode_steps):
        modify(ev,t)
        actions=ctrl.select(actors,obs,t,noise=0)
        obs,r,done,info=ev.step(actions)
        totals["reward"]+=r
        for k in totals:
            if k!="reward": totals[k]+=info[k]
        trace.append([t,info["net_carbon"],info["energy"],info["violation"],
                      info["pipe_pressure"],info["purity"],info["bhp"],info["linepack"]])
    row={"Scenario":name,**totals}
    return row,pd.DataFrame(trace,columns=["Step","NetCarbon","Energy","Violation","PipelinePressure","Purity","BHP","Linepack"])

scenarios=[
    ("Nominal", lambda e,t: None),
    ("Feed disturbance", lambda e,t: setattr(e,"feed", e.feed*(1.18 if 70<=t<120 else 1.0))),
    ("High electricity price", lambda e,t: setattr(e,"elec_price", 125 if 60<=t<160 else e.elec_price)),
    ("Sensor/process noise", lambda e,t: setattr(e,"impurity", np.clip(e.impurity+np.random.normal(0,.01),.005,.10))),
    ("Compressor outage", lambda e,t: setattr(e,"comp_speed", .60 if 80<=t<110 else e.comp_speed)),
    ("Reduced injectivity", lambda e,t: setattr(e,"injectivity", .65 if 150<=t<190 else e.injectivity)),
]
scenario_rows=[]
scenario_traces={}
for name,fn in scenarios:
    row,tr=scenario_run(name,fn)
    scenario_rows.append(row); scenario_traces[name]=tr
scenario_df=pd.DataFrame(scenario_rows)
scenario_df.to_csv(os.path.join(OUT_DIR,"scenario_results.csv"),index=False)
display(scenario_df)


## 10. Ablation study

The manuscript identifies the independent safety filter, multi-timescale coordination and centralized multi-agent coordination as core methodological elements. The following ablations disable one mechanism at a time in the same simulator so their effect can be quantified. These are implementation-level ablations, not claims of experimentally reported values from the manuscript.

In [ ]:
# ============================================================
# 10. ABLATION EVALUATION
# ============================================================
def run_ablation(mode, episodes=4):
    rows=[]
    for e in range(episodes):
        ev=IntegratedCCUSEnv(seed=2000+e)
        obs=ev.reset()
        ctrl=MultiTimescaleController()
        totals={"reward":0,"net_carbon":0,"energy":0,"cost":0,"violation":0,"correction":0}
        for t in range(CFG.episode_steps):
            if mode=="NoSafetyFilter":
                actions=ctrl.select(actors,obs,t,noise=0)
                # bypass safety by making projection identity
                original=ev.safety_filter
                ev.safety_filter=lambda acts:(acts,{"pipe_pressure_pred":ev.pipe_p,
                                                   "purity_pred":ev.purity,
                                                   "bhp_pred":ev.bhp,
                                                   "correction":0.0})
                obs,r,done,info=ev.step(actions)
                ev.safety_filter=original
            elif mode=="SingleTimescale":
                actions=[]
                for i in range(3):
                    x=torch.tensor(obs[i],dtype=torch.float32,device=DEVICE).unsqueeze(0)
                    with torch.no_grad(): aa=actors[i](x).cpu().numpy()[0]
                    actions.append(np.clip(aa,-1,1))
                obs,r,done,info=ev.step(actions)
            else:
                actions=ctrl.select(actors,obs,t,noise=0)
                obs,r,done,info=ev.step(actions)
            totals["reward"]+=r
            for k in ["net_carbon","energy","cost","violation","correction"]: totals[k]+=info[k]
        rows.append(totals)
    return pd.Series({
        "Variant":mode,
        "Return":np.mean([x["reward"] for x in rows]),
        "Net CO2":np.mean([x["net_carbon"] for x in rows]),
        "Energy":np.mean([x["energy"] for x in rows]),
        "Cost":np.mean([x["cost"] for x in rows]),
        "Violation":np.mean([x["violation"] for x in rows]),
        "Correction":np.mean([x["correction"] for x in rows])
    })

ablation=pd.DataFrame([run_ablation("Full SF-MT-MARL"),
                       run_ablation("NoSafetyFilter"),
                       run_ablation("SingleTimescale")])
ablation.to_csv(os.path.join(OUT_DIR,"ablation_results.csv"),index=False)
display(ablation)


In [ ]:
# ============================================================
# 11. VISUALIZATIONS
# ============================================================
plt.figure(figsize=(9,5))
plt.plot(train_df["Episode"],train_df["Return"],marker="o")
plt.xlabel("Episode",fontweight="bold"); plt.ylabel("Episode Return",fontweight="bold")
plt.title("SF-MT-MARL Training Return",fontweight="bold")
plt.grid(alpha=.25); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,"training_return.png"),dpi=300)
plt.show()

plt.figure(figsize=(9,5))
plt.plot(train_df["Episode"],train_df["MeanViolation"],marker="o")
plt.xlabel("Episode",fontweight="bold"); plt.ylabel("Mean Safety Violation",fontweight="bold")
plt.title("Training Safety Constraint Violation",fontweight="bold")
plt.grid(alpha=.25); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,"training_violation.png"),dpi=300)
plt.show()

plt.figure(figsize=(9,5))
x=np.arange(len(comparison))
plt.bar(x-.18,comparison["Net CO2 retained"],width=.36,label="Net CO2 retained")
plt.bar(x+.18,comparison["Energy"],width=.36,label="Energy")
plt.xticks(x,comparison["Controller"]); plt.legend()
plt.ylabel("Mean episode value",fontweight="bold")
plt.title("Controller Comparison",fontweight="bold")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR,"controller_comparison.png"),dpi=300); plt.show()

nom=scenario_traces["Nominal"]
plt.figure(figsize=(9,5))
plt.plot(nom["Step"],nom["PipelinePressure"],label="Pipeline pressure")
plt.axhline(120,linestyle="--",label="Upper limit")
plt.axhline(70,linestyle="--",label="Lower limit")
plt.xlabel("Step",fontweight="bold"); plt.ylabel("Pressure (bar)",fontweight="bold")
plt.title("Nominal Pipeline Pressure",fontweight="bold"); plt.legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR,"pipeline_pressure.png"),dpi=300); plt.show()

plt.figure(figsize=(9,5))
plt.plot(nom["Step"],nom["BHP"],label="Bottom-hole pressure")
plt.axhline(285,linestyle="--",label="Safety limit")
plt.xlabel("Step",fontweight="bold"); plt.ylabel("BHP (bar)",fontweight="bold")
plt.title("Reservoir Pressure Safety",fontweight="bold"); plt.legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR,"bhp_safety.png"),dpi=300); plt.show()


In [ ]:
# ============================================================
# 12. PERCENT CHANGE + METRICS TABLE
# ============================================================
p=comparison.iloc[0]
b=comparison.iloc[1]

summary_metrics=pd.DataFrame({
    "Metric":[
        "Net CO2 retention improvement (%)",
        "Energy reduction (%)",
        "Operating cost reduction (%)",
        "Safety violation reduction (%)",
        "Vented CO2 reduction (%)"
    ],
    "Value":[
        100*(p["Net CO2 retained"]-b["Net CO2 retained"])/max(abs(b["Net CO2 retained"]),1e-9),
        100*(b["Energy"]-p["Energy"])/max(abs(b["Energy"]),1e-9),
        100*(b["Operating cost"]-p["Operating cost"])/max(abs(b["Operating cost"]),1e-9),
        100*(b["Safety violation"]-p["Safety violation"])/max(abs(b["Safety violation"]),1e-9),
        100*(b["Vented CO2"]-p["Vented CO2"])/max(abs(b["Vented CO2"]),1e-9)
    ]
})
summary_metrics.to_csv(os.path.join(OUT_DIR,"summary_metrics.csv"),index=False)
display(summary_metrics)

# Save trained networks
for i in range(3):
    torch.save(actors[i].state_dict(),os.path.join(OUT_DIR,f"actor_{i}.pth"))
    torch.save(critics[i].state_dict(),os.path.join(OUT_DIR,f"critic_{i}.pth"))

with open(os.path.join(OUT_DIR,"config.json"),"w") as f:
    json.dump({k:v for k,v in CFG.__dict__.items()},f,indent=2)

print("Saved results to:",OUT_DIR)
print(os.listdir(OUT_DIR))


## 13. Reproducible interpretation

The notebook operationalizes the manuscript's stated loop: subsystem observations → multi-agent suggested actions → multi-timescale synchronization → inter-stage flow coordination → safety projection → environment response → shared carbon/economic/safety reward → centralized critic/actor updates. The safety layer treats RL actions as suggestions and projects them before implementation, matching the manuscript's safety-filter description. fileciteturn0file0L647-L664

The reward includes retained CO₂, venting, energy-related emissions, oil production, cost, constraint violations and control correction, following the manuscript's joint objective formulation. fileciteturn0file0L665-L679

For a publication-grade experiment, replace the synthetic surrogate equations and generated disturbances with calibrated CCSI²/process-simulator data and the specified NETL SACROC reservoir model, while retaining the learning/control architecture.